# 🏀 NBA Lakehouse — Notebook 1: Raw → Bronze

## Overview
This notebook implements the **first layer** of the Medallion Architecture: ingesting raw CSV files from a Databricks Unity Catalog Volume and writing them as **Delta tables** in the Bronze schema.

## Architecture
```
Raw Files (Volume)  →  Bronze Layer (Delta Tables)
     CSV                    Unmodified, schema-inferred
```

## Data Source
- **Dataset:** NBA Players & Teams Stats (basketball-reference.com via Kaggle)
- **Coverage:** 1947–2026 (we filter to 2000+ in Silver)
- **Volume path:** `/Volumes/nba_lakehouse_catalog/raw/raw_files/`

## Files Ingested
| File | Description | Rows |
|------|-------------|------|
| Player Per Game.csv | Per-game stats for every player-season | 33,278 |
| Advanced.csv | Advanced metrics (PER, WS, BPM, VORP) | 33,278 |
| Team Stats Per Game.csv | Team-level per-game stats | 1,907 |
| Team Summaries.csv | Team season summaries with ratings | 1,907 |
| Player Play By Play.csv | Play-by-play derived stats | 18,193 |
| Player Career Info.csv | Player biographical & career info | 5,396 |

## Design Decisions
- **No transformations** in Bronze — raw data is preserved as-is for full lineage
- **Delta format** used from the start for ACID compliance and time travel
- **inferSchema=True** to automatically detect column types from CSV content
- **mode=overwrite** allows re-running the notebook idempotently

## Step 1 — Verify Raw Files
Before writing to Bronze, we verify all 6 source files are accessible from the Unity Catalog Volume and check their row/column counts. This acts as a data availability check before ingestion begins.

In [0]:
# Define the path to the Unity Catalog Volume where raw CSVs are stored
# Format: /Volumes/<catalog>/<schema>/<volume>/
raw_path = "/Volumes/nba_lakehouse_catalog/raw/raw_files/"

# List of all 6 source files to be ingested
files = [
    "Player Per Game.csv",       # Per-game player statistics
    "Advanced.csv",              # Advanced analytics metrics
    "Team Stats Per Game.csv",   # Team-level per-game stats
    "Team Summaries.csv",        # Team season summaries
    "Player Play By Play.csv",   # Play-by-play derived metrics
    "Player Career Info.csv"     # Player biographical information
]

# Verify each file is readable and print its dimensions
# This is a lightweight check — we read without inferSchema for speed
for f in files:
    df = spark.read.option("header", True).csv(f"{raw_path}{f}")
    print(f"✓ {f} — {df.count()} rows, {len(df.columns)} cols")

print("\nAll files readable!")

## Step 2 — Ingest to Bronze Layer

Each CSV file is read with `inferSchema=True` (so Spark auto-detects integer/double/string types) and written as a **Delta table** into `nba_lakehouse_catalog.bronze`.

**Key points:**
- `inferSchema=True` — Spark scans the data to assign correct types automatically
- `format("delta")` — writes in Delta Lake format, enabling ACID transactions and time travel
- `mode("overwrite")` — makes the pipeline idempotent; safe to re-run without duplicates
- `saveAsTable()` — registers the table in Unity Catalog for governance and discoverability

In [0]:
# Mapping of Delta table names to their source CSV files
# Table names follow snake_case convention for consistency
tables = {
    "player_per_game":     "Player Per Game.csv",
    "advanced":            "Advanced.csv",
    "team_stats_per_game": "Team Stats Per Game.csv",
    "team_summaries":      "Team Summaries.csv",
    "player_play_by_play": "Player Play By Play.csv",
    "player_career_info":  "Player Career Info.csv",
}

print("Writing to Bronze layer...\n")

for table_name, file_name in tables.items():
    # Read CSV with header and automatic schema inference
    # inferSchema triggers an extra scan pass but ensures correct types
    df = spark.read \
        .option("header", True) \
        .option("inferSchema", True) \
        .csv(f"{raw_path}{file_name}")

    # Write to Unity Catalog Bronze schema as a managed Delta table
    # Full table path: catalog.schema.table
    bronze_table = f"nba_lakehouse_catalog.bronze.{table_name}"
    df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(bronze_table)

    print(f"✓ {table_name} — {df.count()} rows written to bronze")

print("\nBronze layer complete!")

## Summary

✅ **Bronze layer complete** — 6 Delta tables registered in `nba_lakehouse_catalog.bronze`

| Table | Rows | Status |
|-------|------|--------|
| player_per_game | 33,278 | ✅ |
| advanced | 33,278 | ✅ |
| team_stats_per_game | 1,907 | ✅ |
| team_summaries | 1,907 | ✅ |
| player_play_by_play | 18,193 | ✅ |
| player_career_info | 5,396 | ✅ |

**Next:** Run `02_Bronze_To_Silver` to clean, validate and enrich the data.